# Pothole Detection (5,113 Images) - YOLOv8 Training on GPU

This notebook trains **YOLOv8s** on the **local 5,113-image pothole-only dataset** (RDD2022 + Hugging Face sets, de-duplicated) using free GPU on Google Colab / Kaggle.

### Pipeline Overview:
1. **Environment:** Verify NVIDIA GPU
2. **Dataset:** Upload `pothole_5k.zip` (from `datasets/pothole_5k.zip` on your PC)
3. **Training:** YOLOv8s, Mosaic + HSV augmentation, up to 50 epochs with early stopping
4. **Evaluation:** Precision, recall, mAP@0.5 and mAP@0.5:0.95 on the held-out test split
5. **Export:** Download `best.pt` into `backend/weights/pothole_det.pt`

In [ ]:
# Step 1: Verify GPU Acceleration
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print(" Please go to Runtime -> Change runtime type -> Select T4 GPU!")

In [ ]:
# Step 2: Install Ultralytics
!pip install -q ultralytics

In [ ]:
# Step 3: Extract and prepare the dataset
import os, glob, zipfile

# 1. Check if already extracted
if not os.path.exists('datasets/pothole_5k/images'):
    os.makedirs('datasets', exist_ok=True)
    
    # 2. Check if pothole_5k.zip is already uploaded in Colab files (left sidebar)
    if os.path.exists('pothole_5k.zip'):
        print('Found pothole_5k.zip in Colab files! Unzipping...')
        with zipfile.ZipFile('pothole_5k.zip') as z:
            z.extractall('datasets')
    elif os.path.exists('/content/pothole_5k.zip'):
        print('Found /content/pothole_5k.zip! Unzipping...')
        with zipfile.ZipFile('/content/pothole_5k.zip') as z:
            z.extractall('datasets')
    else:
        # 3. If not found in files, prompt for upload
        print('pothole_5k.zip not found in files. Please upload it...')
        from google.colab import files
        up = files.upload()
        with zipfile.ZipFile(list(up.keys())[0]) as z:
            z.extractall('datasets')

# Rewrite data.yaml with absolute Colab path
with open('datasets/pothole_5k/data.yaml', 'w') as f:
    f.write('path: /content/datasets/pothole_5k\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: pothole\n')

# Verify counts
n = {s: len(glob.glob(f'datasets/pothole_5k/images/{s}/*')) for s in ('train', 'val', 'test')}
print(f' Dataset Ready! Splits: {n}')
print(f' TOTAL: {sum(n.values())} pothole images')

In [ ]:
# Step 4: Train YOLOv8s for 50 Epochs
from ultralytics import YOLO

# Load pre-trained YOLOv8s backbone for transfer learning
model = YOLO('yolov8s.pt')

# Train with GPU acceleration
results = model.train(
    data='datasets/pothole_5k/data.yaml',
    epochs=50,
    imgsz=640,
    batch=32,            # 32 images per batch on T4 GPU (16GB VRAM)
    device=0,            # Use GPU
    patience=15,         # Early stopping if mAP doesn't improve for 15 epochs
    save=True,
    project='pothole_training_5k',
    name='yolov8s_5k_run',
    mosaic=1.0,          # Combine 4 images into 1 for scale invariance
    hsv_h=0.015,         # Color jitter (handles varying sunlight)
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5           # Random horizontal flip
)

In [ ]:
# Step 5: Evaluate Model on Held-out Test Split
test_metrics = model.val(data='datasets/pothole_5k/data.yaml', split='test')
print("="*50)
print("FINAL TEST METRICS (5,113 Image Trained Model):")
print(f"  Precision:    {test_metrics.box.mp:.4f}")
print(f"  Recall:       {test_metrics.box.mr:.4f}")
print(f"  mAP@0.5:      {test_metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {test_metrics.box.map:.4f}")
print("="*50)

In [ ]:
# Step 6: Display Training Curves & Confusion Matrix
from IPython.display import Image, display
import os

run_dir = 'pothole_training_5k/yolov8s_5k_run'
for plot in ['results.png', 'confusion_matrix.png', 'BoxPR_curve.png', 'BoxF1_curve.png']:
    p = os.path.join(run_dir, plot)
    if os.path.exists(p):
        print(f"Displaying {plot}:")
        display(Image(filename=p))

In [ ]:
# Step 7: Download your trained weights directly to your computer
from google.colab import files
best_weights = 'pothole_training_5k/yolov8s_5k_run/weights/best.pt'
print(f"Downloading {best_weights} ...")
files.download(best_weights)
print("Replace backend/weights/pothole_det.pt with this downloaded best.pt!")